# Simulando lentes gravitacionales fuertes

![](https://i.imgur.com/ktoS9gT.png)

## CdeCMx 2025 - GTO2: Cosmic distortions.

### By: Gabriel Missael Barco & Zaid de Anda Mariscal

Un poco complicado instalar caustics... para ello:
1. Dar click a la siguiente celda
2. Te va a salir un mensaje de re-iniciar el kernel, dale que si!

In [ ]:
%pip install astropy
%pip install "astropy[recommended]"
%pip install caustics

In [ ]:
# Para mostrar figuras
import matplotlib.pyplot as plt

# Software de simulación
import caustics

# Para tensores y operaciones matemáticas
import torch
import numpy as np

# Para cargar el dataset
import os

# Cosas interactivas
from ipywidgets import interact, FloatSlider, Layout, interactive_output
import ipywidgets as w

# Uso de la GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def make_slider(val, vmin, vmax, step, desc):
    s = w.FloatSlider(value=val, min=vmin, max=vmax, step=step,
                      description=desc,
                      readout=True)
    s.style.description_width = '125px'
    return s

Primero, tenemos que definir algunas caracteristicas de nuestro telescopio y la observación que realizamos!

In [ ]:
# Tamaño de la imagen
obs_fov = 12
source_fov = 3.5

# Resolución de la imagen
obs_pixels = 128
source_pixels = 64

# Creamos matrices de pixeles
pixelscale_source = source_fov / source_pixels
thx_source, thy_source = caustics.utils.meshgrid(pixelscale_source, source_pixels, source_pixels)

pixelscale_obs = obs_fov / obs_pixels
thx_obs, thy_obs = caustics.utils.meshgrid(pixelscale_obs, obs_pixels, obs_pixels)


In [ ]:
empty = torch.zeros(1600, 1600)

obs_window = empty.clone()
obs_window[200:1400, 200:1400] = 1.0

source_window = empty.clone()
source_window[625:975, 625:975] = 1.0

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(obs_window, origin='lower', cmap='gray')
ax.imshow(source_window, origin='lower', cmap='gray', alpha=0.5)
ax.set_title('Ventanas del Observador y de la Fuente')

ax.set_xticks([(x)*100 for x in range(16)])
ax.set_yticks([(y)*100 for y in range(16)])
ax.set_xticklabels([str(x-8) for x in range(16)])
ax.set_yticklabels([str(y-8) for y in range(16)])
ax.set_xlabel('Ángulo (arcsec)')
ax.set_ylabel('Ángulo (arcsec)')
plt.show()

In [ ]:
# Definimos la cosmología con un universo plano
cosmology = caustics.FlatLambdaCDM()

La intensidad de la luz de la galaxia de fondo puede ser modelada con una función paramétrica, como un perfil Sersic:

$$I(R)=I_0\exp\!\left[-b_n\!\left(\frac{R^{1/n}}{R_e}-1\right)\right]$$

In [ ]:
# La fuente es una galaxia con perfil de luz Sersic
src = caustics.Sersic(name="source",
                      x0=0.0, # Posición de la fuente en el plano del cielo
                      y0=0.0,
                      q=0.5, # Eje mayor de la fuente
                      phi=-0.985, # Orientación del eje mayor de la fuente (en radianes)
                      n=1.3, # Índice de Sersic
                      Re=1.0, # Radio efectivo de la fuente
                      Ie=5.0) # Intensidad central de la fuente

In [ ]:
# Podemos evaluar la intensidad de la fuente en el plano del cielo
light = src.brightness(thx_source, thy_source)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(light, origin='lower', cmap='inferno')
ax.set_title('Perfil de Luz de la Fuente')

plt.axis('off')
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Sliders for the Sérsic source
# ------------------------------------------------------------------
q_src   = make_slider(0.5, 0.1, 1.0, 0.1, 'Eje (q)')
phi_src = make_slider(-0.985, -3.14, 3.14, 0.01, 'Orientación (rad)')
n_src   = make_slider(1.3, 0.5, 3.0, 0.1, 'Índice n')
Re_src  = make_slider(1.0, 0.1, 5.0, 0.1, 'Radio efectivo Rₑ')

source_box = w.GridBox(children=[q_src, phi_src, n_src, Re_src],
                       layout=Layout(grid_template_columns='repeat(2, 240px)',
                                     grid_gap='5px 25px'))

ui = w.Accordion(children=[source_box])
ui.set_title(0, 'Fuente — perfil Sérsic')

# ------------------------------------------------------------------
def plot_source(q, phi, n, Re):
    src = caustics.Sersic(name="source",
                          x0=0.0, y0=0.0,
                          q=q, phi=phi, n=n, Re=Re, Ie=5)

    light = src.brightness(thx_source, thy_source)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(light, origin='lower', cmap='inferno')

    ax.set_xticks([x*16 for x in range(4)])
    ax.set_yticks([y*16 for y in range(4)])
    ax.set_xticklabels([f'{x*16*pixelscale_source - 1.75:.2f}' for x in range(4)])
    ax.set_yticklabels([f'{y*16*pixelscale_source - 1.75:.2f}' for y in range(4)])
    ax.set_xlabel('Δx [arcsec]')
    ax.set_ylabel('Δy [arcsec]')
    plt.show()


# ------------------------------------------------------------------
# Wire widgets → callback and display everything
# ------------------------------------------------------------------
controls = dict(q=q_src, phi=phi_src, n=n_src, Re=Re_src)
out = interactive_output(plot_source, controls)

display(ui, out)

$$\Sigma(\boldsymbol{\xi}) =\frac{v^{2}\sqrt{{f}}}{2G}\frac{1}{\sqrt{\xi_{1}^{2}+f^{2}\xi_{2}^{2}}}$$

In [ ]:
# Lente gravitacional

# Tiene un perfil de masa SIE (Ellipsoidal Isothermal Ellipsoid)
sie = caustics.SIE(cosmology=cosmology,
                   name="lens",
                   z_l=0.5, # Redshift del lente
                   z_s=1.0, # Redshift de la fuente
                   x0=-0.2, # Posición del centro del lente en el plano del cielo
                   y0=0.0,
                   q=0.7, # Eje mayor del elipsoide
                   phi=1.5708, # Orientación del eje mayor (en radianes)
                   Rein=1.5) # Radio de Einstein

# y tiene un perfil de luz
lnslt = caustics.Sersic(name="lenslight",
                        x0=-0.2, # Posición del centro de la luz del lente en el plano del cielo
                        y0=0.0,
                        q=0.7, # Eje mayor de la luz del lente
                        phi=1.5708, # Orientación del eje mayor de la luz del lente (en radianes)
                        n=1.0, # Índice de Sersic
                        Re=0.7, # Radio efectivo de la luz del lente
                        Ie=10.0) # Intensidad central de la luz del lente

In [ ]:
conv = sie.convergence(thx_obs, thy_obs)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(torch.log(conv), origin='lower', cmap='inferno')

ax.set_title('Log-Convergencia de la lente gravitacional')

ax.set_xticks([(x)*32 for x in range(5)])
ax.set_yticks([(y)*32 for y in range(5)])
ax.set_xticklabels([str(x*32*pixelscale_obs - 6) for x in range(5)])
ax.set_yticklabels([str(y*32*pixelscale_obs - 6) for y in range(5)])

plt.show()

In [ ]:
# ---------------------------------------------------------------
# 1.  Sliders para la lente SIE
# ---------------------------------------------------------------
q_lens   = make_slider(0.7, 0.1, 1.0, 0.1, 'Eje (q)')
phi_lens = make_slider(1.5708, -3.14, 3.14, 0.01, 'Orientación (rad)')
Rein     = make_slider(1.7, 0.1, 5.0, 0.1, 'Radio Einstein Rₑ')

lens_box = w.GridBox(children=[q_lens, phi_lens, Rein],
                     layout=Layout(grid_template_columns='repeat(2, 240px)',
                                   grid_gap='6px 25px'))

ui = w.Accordion(children=[lens_box])
ui.set_title(0, 'Lente (SIE)')

# ---------------------------------------------------------------
def plot_lens(q, phi, Rein):
    sie = caustics.SIE(cosmology=cosmology,
                       name="lens",
                       z_l=0.5, z_s=1.0,
                       x0=-0.2, y0=0.0,
                       q=q, phi=phi, Rein=Rein)

    conv = sie.convergence(thx_obs, thy_obs)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(torch.log(conv), origin='lower', cmap='inferno')
    ax.set_title('Log-Convergencia de la lente gravitacional')

    # rejilla en unidades físicas
    ax.set_xticks([x*32 for x in range(5)])
    ax.set_yticks([y*32 for y in range(5)])
    ax.set_xticklabels([f'{x*32*pixelscale_obs - 6:.1f}' for x in range(5)])
    ax.set_yticklabels([f'{y*32*pixelscale_obs - 6:.1f}' for y in range(5)])
    ax.set_xlabel('Δx [arcsec]')
    ax.set_ylabel('Δy [arcsec]')
    plt.show()

# ---------------------------------------------------------------
# 3.  Vínculo widgets  →  función y display
# ---------------------------------------------------------------
controls = dict(q=q_lens, phi=phi_lens, Rein=Rein)
out = interactive_output(plot_lens, controls)

display(ui, out)


In [ ]:
# Simulamos la lente gravitacional con una cuadrícula de píxeles
sim = caustics.LensSource(lens=sie, source=src, lens_light=lnslt, pixelscale=0.05, pixels_x=100)

In [ ]:
# Ploteamos la imagen de la lente gravitacional
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(sim(),
           origin='lower',
           cmap='inferno')
ax.axis('off')
plt.show()

Challenge: ¿Cómo crear un Einstein Ring?

In [ ]:
# -----  Lens (SIE) sliders  -----
x0_lens  = make_slider(-0.2, -1, 1, .01, r'x₀ lente')
y0_lens  = make_slider( 0.0, -1, 1, .01, r'y₀ lente')
q_lens   = make_slider( 0.7,  .1, 1, .01, 'Eje lente')
phi_lens = make_slider( 1.57, -3.14, 3.14, .01, 'Orient. lente')
Rein     = make_slider( 1.7,  .1, 5,  .1,  'Rₑ Einstein')

lens_box = w.GridBox(children=[x0_lens, y0_lens, q_lens, phi_lens, Rein],
                     layout=Layout(grid_template_columns='repeat(2, 240px)',
                                   grid_gap='6px 10px'))

# -----  Source sliders  -----
x0_src  = make_slider(0.0, -1, 1, .01, r'x₀ fuente')
y0_src  = make_slider(0.0, -1, 1, .01, r'y₀ fuente')
q_src   = make_slider(0.5, .1, 1, .01, 'Eje fuente')
phi_src = make_slider(-0.985, -3.14, 3.14, .01, 'Orient. fuente')
Re_src  = make_slider(1.0, 0.1, 5, .1, 'Rₑ fuente')
n_src   = make_slider(1.3, 0.5, 3, .1, 'Índice n fuente')

src_box = w.GridBox(children=[x0_src, y0_src, q_src, phi_src,
                              Re_src, n_src],
                    layout=lens_box.layout)

# -----  Lens-light sliders  -----
n_L   = make_slider(1.0, 0.5, 3, .1, 'Índice luz lente')
Re_L  = make_slider(1.0, 0.1, 5, .1, 'Rₑ luz lente')

light_box = w.GridBox(children=[n_L, Re_L],
                      layout=lens_box.layout)
ui = w.Accordion(children=[lens_box, src_box, light_box])
ui.set_title(0, 'Lente (SIE)')
ui.set_title(1, 'Fuente')
ui.set_title(2, 'Luz del lente')


# -------------------------------------------------
# 3.  Plotting callback (unchanged physics code)
# -------------------------------------------------
def plot_lens_source(**p):
    sie = caustics.SIE(cosmology=cosmology,
                       name="lens", z_l=0.5, z_s=1.0,
                       x0=p['x0_lens'], y0=p['y0_lens'],
                       q=p['q_lens'],  phi=p['phi_lens'],
                       Rein=p['Rein'])

    src = caustics.Sersic(name="source",
                          x0=p['x0_src'], y0=p['y0_src'],
                          q=p['q_src'],  phi=p['phi_src'],
                          n=p['n_src'],  Re=p['Re_src'],
                          Ie=1)

    lnslt = caustics.Sersic(name="lenslight",
                            x0=p['x0_lens'], y0=p['y0_lens'],
                            q=p['q_lens'],  phi=p['phi_lens'],
                            n=p['n_L'],     Re=p['Re_L'],
                            Ie=4)

    sim = caustics.LensSource(lens=sie, source=src, lens_light=lnslt,
                              pixelscale=pixelscale_obs, pixels_x=obs_pixels)

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(src.brightness(thx_source, thy_source), origin='lower', cmap='inferno')
    ax[0].set_title('Perfil de Luz de la Fuente'); ax[0].axis('off')

    ax[1].imshow(torch.log(sie.convergence(thx_obs, thy_obs)), origin='lower', cmap='inferno')
    ax[1].set_title('Log-Convergencia de la lente'); ax[1].axis('off')

    ax[2].imshow(sim(), origin='lower', cmap='inferno')
    ax[2].set_title('Imagen de la lente'); ax[2].axis('off')
    plt.show()


# -------------------------------------------------
# 4.  Link widgets → callback and display
# -------------------------------------------------
controls = dict(x0_lens=x0_lens, y0_lens=y0_lens, q_lens=q_lens,
                phi_lens=phi_lens, Rein=Rein,
                x0_src=x0_src, y0_src=y0_src, q_src=q_src,
                phi_src=phi_src, Re_src=Re_src, n_src=n_src,
                n_L=n_L, Re_L=Re_L)

out = interactive_output(plot_lens_source, controls)

display(ui, out)


In [ ]:
lens = sim()
plt.imshow(lens, origin='lower', cmap='inferno')

In [ ]:
import scipy

n= np.zeros((21,21))
n[10,10] = 1
k = scipy.ndimage.gaussian_filter(n,sigma=2)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(n, origin='lower', cmap='inferno')
ax[0].set_title('Fuente puntual')
ax[1].imshow(k, origin='lower', cmap='inferno')
ax[1].set_title('Kernel Gaussiano con σ=2')


In [ ]:
# Convolve lens with a Gaussian kernel
from scipy.ndimage import gaussian_filter
lens_convolved = torch.tensor(gaussian_filter(lens.numpy(), sigma=2))

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(lens.numpy(), origin='lower', cmap='inferno')
ax[0].set_title('Lente original')
ax[1].imshow(lens_convolved, origin='lower', cmap='inferno')
ax[1].set_title('Lente con PSF')
plt.show()

In [ ]:
# Poission noise
lens_noise = np.random.poisson(lens_convolved.numpy() * 2) / 2.0

# Plot
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(lens.numpy(), origin='lower', cmap='inferno')
ax[0].set_title('Lente original')
ax[1].imshow(lens_convolved, origin='lower', cmap='inferno')
ax[1].set_title('Lente con PSF')
ax[2].imshow(lens_noise, origin='lower', cmap='inferno')
ax[2].set_title('Lente con ruido Poisson')
plt.show()

In [ ]:
# Guassian noise
lens_noise_final = torch.tensor(lens_noise) + torch.randn_like(torch.tensor(lens_noise))*4

# Plot
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(lens.numpy(), origin='lower', cmap='inferno')
ax[0].set_title('Lente original')
ax[1].imshow(lens_convolved, origin='lower', cmap='inferno')
ax[1].set_title('Lente con PSF')
ax[2].imshow(lens_noise, origin='lower', cmap='inferno')
ax[2].set_title('Lente con ruido Poisson')
ax[3].imshow(lens_noise_final, origin='lower', cmap='inferno')
ax[3].set_title('Lente con ruido Poisson + Gaussiano')
plt.show()

In [ ]:
# Clean observation vs real observation
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(lens.numpy(), origin='lower', cmap='inferno')
ax[0].set_title('Observación limpia')
ax[1].imshow(lens_noise_final, origin='lower', cmap='inferno')
ax[1].set_title('Observación realista')
plt.show()

# Simulaciones de galaxias

In [ ]:
import gdown

file_id = "1_yYqal8jvobOKzCDzCKOcxiMAfWhFFuA"
url = f"https://drive.google.com/uc?id={file_id}"
output = "galaxy_dataset.npy"
gdown.download(url, output, quiet=False)


In [ ]:
train_set = np.load("/content/galaxy_dataset.npy")

In [ ]:
fig, axs = plt.subplots(4, 8, figsize=(20, 10))
for i, ax in enumerate(axs.flat):
    ax.imshow(train_set[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

# Cool simulator

In [ ]:
import caustics
from caustics import Module, forward
import numpy as np
import torch
from typing import Optional
from copy import copy
from scipy.fft import next_fast_len
from numpy.typing import NDArray
from torch.func import vmap, jacrev
import os
import h5py
from typing import Literal

data_path = "./data"

class SimpleSimulator(Module):
    def __init__(
        self,
        name = "SimpleSimulator",
        source_pixels: int = 64,
        observation_pixels: int = 64,
        observation_fov: float = 12,
        source_fov: float = 5,#6.2432,
        upsample=2,
        z_l: float = 0.5,
        z_s: float = 1.0,
        psf_sigma = None,
        sigma_n = None,
        poisson_rate = None,
        include_lens_light = False,
    ):
        super().__init__()

        self.source_fov = source_fov
        self.observation_fov = observation_fov
        self.source_pixels = source_pixels
        self.observation_pixels = observation_pixels
        self.cosmo = caustics.FlatLambdaCDM(name="cosmo")
        self.source_pixelscale = self.source_fov / self.source_pixels
        self.observation_pixelscale = self.observation_fov / self.observation_pixels
        self.upsample = upsample
        self.psf_sigma = psf_sigma
        self.sigma_n = sigma_n
        self.poisson_rate = poisson_rate
        self.include_lens_light = include_lens_light

        self.lens_epl = caustics.EPL(cosmology=self.cosmo, name="epl",
                                     x0=0.0, y0=0.0)
        self.lens = caustics.SinglePlane(
            name="lensmass", cosmology=self.cosmo, lenses=[self.lens_epl]
        )
        self.source = caustics.Pixelated(
            pixelscale=self.source_pixelscale,
            shape=(source_pixels, source_pixels),
            name="source",
        )
        thx, thy = caustics.utils.meshgrid(
            self.observation_pixelscale / self.upsample,
            self.upsample * observation_pixels,
            dtype=torch.float32,
            device=DEVICE,
        )
        self.thx = thx.to(DEVICE)
        self.thy = thy.to(DEVICE)
        self.lens.z_l = z_l
        self.lens.z_s = z_s

        if self.include_lens_light:
            self.lens_light = caustics.Pixelated(
                pixelscale=self.observation_pixelscale,
                shape=(observation_pixels, observation_pixels),
                name="lenslight",
                x0=0.0, y0=0.0
            )

    @forward
    def __call__(self):
        # Ray-trace to get the lensed positions
        bx, by = self.lens.raytrace(self.thx, self.thy)

        # Evaluate the lensed source brightness at high resolution
        image = self.source.brightness(bx, by)

        if self.psf_sigma is not None:
            mu_convolved = torch.tensor(gaussian_filter(image.cpu().numpy(), sigma=self.psf_sigma*self.upsample))
        else:
            mu_convolved = image

        # Downsample to the desired resolution
        image_ds = (
            torch.nn.functional.avg_pool2d(mu_convolved.unsqueeze(0), self.upsample)
            .squeeze(0)
            .to(DEVICE)
        )
        if self.include_lens_light:
            mu = self.lens_light.brightness(self.thx, self.thy)
            mu = (
                torch.nn.functional.avg_pool2d(mu.unsqueeze(0), self.upsample)
                .squeeze(0)
                .to(DEVICE)
            )
            # Convolve psf
            mu_convolved = torch.tensor(gaussian_filter(mu.cpu().numpy(), sigma=self.psf_sigma*self.upsample))
            image_ds += mu_convolved
        if self.poisson_rate is not None:
            image_ds = torch.poisson((image_ds+image_ds.min()+1e-3) * self.poisson_rate) / self.poisson_rate
        if self.sigma_n is not None:
            image_ds += torch.randn_like(image_ds) * self.sigma_n

        return image_ds

In [ ]:
elements = [
    "q",
    "phi",
    "Rein",
    "t",
    "x0_src",
    "y0_src",
]

# default numeric bounds for each parameter
bounds = {
    "q": (0.4, 1.00),
    "phi": (0.00, np.pi),
    "Rein": (0.1, 3),
    "t": (0.6, 1.4),
    "x0_src": (-1, 1),
    "y0_src": (-1, 1),
}

def sample_lens(n):
    lower_bound = torch.tensor([bounds[element][0] for element in elements], device=DEVICE)
    upper_bound = torch.tensor([bounds[element][1] for element in elements], device=DEVICE)

    dist = torch.distributions.uniform.Uniform(low=lower_bound, high=upper_bound)
    lens_params = dist.sample((n,))
    lens_params = lens_params.to(DEVICE)
    return lens_params


print("Sampled lens parameters:", sample_lens(2))

In [ ]:
lens_samples = sample_lens(32)

sim = SimpleSimulator(psf_sigma=0.5,
                      poisson_rate=400,
                      sigma_n=0.05,
                      include_lens_light=True)

lenses = []
for lens, source in zip(lens_samples, train_set[:32]):
    lens = torch.tensor(lens).flatten()
    source = torch.tensor(source).flatten()
    # Random luz de la lente
    rand_idx = np.random.randint(0, 1000)
    lens_light = train_set[rand_idx].flatten()
    lens_light = torch.tensor(lens_light).flatten()

    x_fwd = torch.cat([lens, source, lens_light])
    lens = sim(x_fwd)
    # Normalizar
    lens = lens / lens.max()
    lenses.append(lens)

fig, axs = plt.subplots(4, 8, figsize=(20, 10))
for i, ax in enumerate(axs.flat):
    ax.imshow(lenses[i].cpu().numpy(), origin='lower')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# RETO: Completar simulaciones de solo galaxias!

galaxies = []

for i in range(32):
    rand_idx = np.random.randint(0, 1000)
    img = train_set[rand_idx]

    # CÓDIGO AQUI - Agregar PSF, ruido poisson, ruido gaussiano, y normalizar
    # img = ?

    galaxies.append(img)

fig, ax = plt.subplots(4, 8, figsize=(20, 10))
for i, ax in enumerate(ax.flat):
    ax.imshow(galaxies[i], origin='lower')
    ax.axis('off')
plt.tight_layout()


In [ ]:
# RETO: Crear un set de 8000 lentes, y otro de 8000 galaxias
# Guardarlo como archivo npy!